#Read csv file using data frame reader API

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config



In [0]:
%run ../00-common/02.BronzeHelper

In [0]:
source_path=f"{landing_folder_path}/{v_batch_id}/drivers.json"
table_name=f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

name_schema = StructType([
  StructField('givenName', StringType(), True),
  StructField('familyName', StringType(), True)])

driver_schema = StructType([
  StructField('driverId', StringType(), True),
  StructField('name', name_schema, True),
  StructField('dateOfBirth', DateType(), True),
   StructField('nationality', StringType(), True),
  StructField('url', StringType(), True)
])
#spark.createDataFrame([], driver_schema).printSchema()
df_drivers = (spark.read.format("json")
               .option('header', True)
               .option('mode', 'FAILFAST')  # strict mode datatype validation
             #  .option('mod', 'PERMISSIVE') # ignore bad records with null value
              # .option('inferSchema', True)  optional incase of schema passing as below
               .schema(driver_schema)
               .load(source_path))

In [0]:
df_drivers.show();

In [0]:
import pyspark.sql.functions as F

df_drivers_final=add_ingestion_metadata(df_drivers)
display(df_drivers_final)

In [0]:
df_drivers_final=df_drivers_final.drop("url")

In [0]:
# (df_drivers_final.write
#       .format("delta")
#       .mode("overwrite")
#       .saveAsTable(table_name))

In [0]:
write_to_bronze(
    input_df=df_drivers_final,
    target_table=table_name,
    batch_id=v_batch_id)

In [0]:
%sql
select * from formula1_incr_catalog.bronze.drivers

In [0]:
df_table=spark.read.table(table_name)
display(df_table)